<a href="https://colab.research.google.com/github/Khaddie100/Data-analysis-projects/blob/main/Pandas_graded_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
df = pd.read_excel('saas.xlsx')

df.head()

,account_id,company,signup_ts,country,seats,mrr,plan,last_active,churned
0,A-1001,Acme Corp,2023-01-05 09:00:00,US,25.0,2500,enterprise,2024-03-01,False
1,A-1002,Beta LLC,2023-02-14 00:00:00,US,5.0,450,pro,2024-03-02,False
2,A-1001,Acme Corp,2023-01-05 09:00:00,US,25.0,2500,enterprise,2024-03-01,False
3,A-1003,Cirrus,1675209600,GB,0.0,0,pro,2024-02-28,False
4,A-1004,Delta Inc,2023-03-20 00:00:00,United States,12.0,1188,pro,2024-03-01,No


### Section A: Mental model

Question 1: Predict the alignment

1. Prediction:
Initially, I expected a + b to concatenate the two Series because it was combining two separate objects. After reviewing the pandas documentation, I realized this assumption was incorrect. In pandas, the + operator performs element-wise arithmetic, not concatenation. It aligns values using the index labels, so only matching labels are added together, while labels that exist in only one Series produce NaN.

In [ ]:
# 2. Actual output
import pandas as pd

a = pd.Series({'jan': 100, 'feb': 200, 'mar': 300})

b = pd.Series({'feb': 10, 'mar': 20, 'apr': 40})

result = a + b

print(result)

apr      NaN
feb    210.0
jan      NaN
mar    320.0
dtype: float64


3. Explanation: Pandas aligns Series using their index labels rather than their row positions. It creates a result containing the union of all labels from both Series (jan, feb, mar, and apr). Values are added only where the labels exist in both Series, giving 210 for feb and 320 for mar. Where a label is missing from one Series (jan or apr), the operation produces NaN, which also causes the resulting Series to use a floating-point data type.

**Question 2: Series or dataframe?**

In [ ]:
# Epression                    Type           Why the difference matters
df['mrr']                     #series         A Series has one dimension, so operations like value_counts(), map(),
                                              # or arithmetic apply directly to that column.

df[['mrr']]                   #Dataframe      A DataFrame preserves the two-dimensional structure,
                                              # making it easier to join with other DataFrames or pass into functions that expect a DataFrame

df.loc[:, 'mrr']              # Series        Since it returns a Series, methods and attributes specific to Series are available,
                                              # and the result has shape (rows,) rather than (rows, 1)

df.loc[:, ['mrr', 'seats']]   # Dataframe     Keeping multiple columns together allows you to perform operations that
                                              # involve both columns while preserving their relationship.


### Section B: Column slicing

Question 3: Four routes to the same column

1.

```
# df[['company', 'seats', 'mrr']]
```
This method is robust to column reordering because it selects columns by their names rather than their positions. However, it would fail if any of the column names were renamed, because the specified labels would no longer exist.

2.

```
# df.loc[:, 'company':'mrr']
```
This method is fragile because .loc selects every column between the start and end labels. If the column order changes, the range may include different columns or exclude the intended ones. If either company or mrr is renamed, the label range will also fail.

3.

```
# df.iloc[:, [1,4,5]]
```
This method is fragile because .iloc uses column positions instead of names. If columns are inserted, removed, or reordered, positions 1, 4, and 5 may no longer correspond to company, seats, and mrr, resulting in the wrong columns being selected.

4.

```
# df.filter(items=['company', 'seats', 'mrr'])
```
This method is not affected by column order because it searches for the specified column names directly. However, like any label-based method, it would fail if one of those column names were renamed.

What single route would you put in a production code and why?

I would use

```
# df[['company', 'seats', 'mrr']]
```

 in production because it is explicit, easy to read, and selects columns by their names rather than their positions. It is not affected by column reordering, and if an expected column has been renamed or removed, it raises an error immediately, making schema changes easier to detect and fix.

Question 4: The inclusive/exclusive trap

In [ ]:
df.loc[:, 'company':'mrr']

# Prediction: .loc will return 5 columns: company, signup_ts, country, seats,

# Rule explanation: .loc uses row and column labels for selection. When a range of labels is provided,
# pandas includes both the starting and ending labels in the result, making label-based slices inclusive.

,company,signup_ts,country,seats,mrr
0,Acme Corp,2023-01-05 09:00:00,US,25.0,2500
1,Beta LLC,2023-02-14 00:00:00,US,5.0,450
2,Acme Corp,2023-01-05 09:00:00,US,25.0,2500
3,Cirrus,1675209600,GB,0.0,0
4,Delta Inc,2023-03-20 00:00:00,United States,12.0,1188
5,Echo,2023-04-01 00:00:00,US,8.0,-99
6,Foxtrot,2023-05-11 00:00:00,us,3.0,270
7,Gamma,2023-06-30 00:00:00,DE,15.0,1800
8,Helio,2023-07-22 00:00:00,US,NaN,540
9,Iris,2023-08-08 00:00:00,US,10.0,900


In [ ]:
df.iloc[:, 1:5]

# Prediction: .iloc will return 4 column from 1 up to 4 excluding 5: company, signup_ts, country, seats

# Rule explanation: .iloc uses integer positions instead of labels. It follows Python's standard slicing behavior,
# where the starting position is included but the ending position is excluded.

,company,signup_ts,country,seats
0,Acme Corp,2023-01-05 09:00:00,US,25.0
1,Beta LLC,2023-02-14 00:00:00,US,5.0
2,Acme Corp,2023-01-05 09:00:00,US,25.0
3,Cirrus,1675209600,GB,0.0
4,Delta Inc,2023-03-20 00:00:00,United States,12.0
5,Echo,2023-04-01 00:00:00,US,8.0
6,Foxtrot,2023-05-11 00:00:00,us,3.0
7,Gamma,2023-06-30 00:00:00,DE,15.0
8,Helio,2023-07-22 00:00:00,US,NaN
9,Iris,2023-08-08 00:00:00,US,10.0


### Section c: Diagnosis and cleaning

Question 5: The count is the question

| Problem | Category | Fix? | Reason |
|---|---|---|---|
| A-1001 appears more than once | What is one row? | Investigate | The repeated account ID may represent a duplicate or a legitimate repeated account record. I would need to understand what one row represents before removing either record. |
| PRO vs pro | What should each column be? | Yes | These appear to represent the same plan but use inconsistent capitalization, so I would standardize the capitalization to avoid treating them as separate categories during analysis. |
| A-1002 appears more than once | What is one row? | Investigate | The two records contain different seats and MRR values, so they may represent an account update rather than an exact duplicate. |
| United States instead of US | What should each column be? | Yes | Both values appear to represent the same country, so I would standardize the country representation. |
| us instead of US	| What should each column be?	| Yes	| The value differs only in capitalization, so I would standardize it to the same format as the other US records. |
| MRR of -99	| What is missing or lying?	| Investigate	| Negative MRR is unusual, but I would not automatically replace it because the value could represent a special business condition or coding convention. I would confirm its meaning first. |
| Seats of 0	| What is missing or lying?	| Investigate	| Zero seats is unusual for a paying SaaS account, but the data alone does not prove that it is invalid. I would confirm the business rule before changing it. |
| MRR of 0	| What is missing or lying?	| Investigate	| Zero revenue could be unusual for an enterprise account, but it could also represent a legitimate business situation. I would investigate before changing it. |
| last_active = 2099-01-01	| What is missing or lying?	| Yes/flag	| The date is syntactically valid but is implausible in the context of the rest of the dataset. I would flag it for correction or verification rather than treating it as an ordinary date. |
| Missing company |	What is missing or lying?	| Keep/Investigate	| The company name is missing, but the account may still be useful for analyses that do not require the company name. I would try to recover the value rather than automatically deleting the entire row. |
| Missing seats	| What is missing or lying?	| Investigate	| Seats is a measure needed for calculations such as revenue per seat. I would investigate or recover the value before using this row in analyses that depend on seats. |
| starter plan	| What should each column be?	| No	| Although it occurs only once, starter is a plausible subscription tier. The dataset provides no evidence that only pro and enterprise are valid plans, so I would leave it unchanged.

Legitimate data that I would leave alone

The starter value in the plan column looks unusual because it occurs only once, but I would not classify it as dirty data. A SaaS business can reasonably have different subscription tiers, and there is no information in the dataset stating that starter is an invalid plan. Therefore, I would leave it alone unless the business specification confirmed that only pro and enterprise were valid plans.

Question 6: The duplicate that isn't

PREDICTION: The first code is df = df.drop_duplicates() removes only completely identical rows. Therefore, I predicted that neither repeated account would be fully resolved: A-1001 differs in the churned value (False vs 0), while A-1002 differs in seats, mrr, and last_active


In [ ]:
# Actual code
df[df['account_id'].isin(['A-1001', 'A-1002'])]

,account_id,company,signup_ts,country,seats,mrr,plan,last_active,churned
0,A-1001,Acme Corp,2023-01-05 09:00:00,US,25.0,2500,enterprise,2024-03-01,False
1,A-1002,Beta LLC,2023-02-14 00:00:00,US,5.0,450,pro,2024-03-02,False
2,A-1001,Acme Corp,2023-01-05 09:00:00,US,25.0,2500,enterprise,2024-03-01,False
12,A-1002,Beta LLC,2023-02-14 00:00:00,US,6.0,540,pro,2024-03-06,False


Flaw found:
drop_duplicates() checks the complete row by default rather than determining whether duplicate account_id values represent duplicate records. Therefore, it is not appropriate for resolving the A-1002 repeat, which contains a genuine update.

In [ ]:
# Replacement code
df = df.sort_values('last_active')
df = df.drop_duplicates(subset='account_id', keep='last')
df

,account_id,company,signup_ts,country,seats,mrr,plan,last_active,churned
7,A-1007,Gamma,2023-06-30 00:00:00,DE,15.0,1800,enterprise,2023-06-15,True
3,A-1003,Cirrus,1675209600,GB,0.0,0,pro,2024-02-28,False
2,A-1001,Acme Corp,2023-01-05 09:00:00,US,25.0,2500,enterprise,2024-03-01,False
4,A-1004,Delta Inc,2023-03-20 00:00:00,United States,12.0,1188,pro,2024-03-01,No
10,A-1010,Juno,2023-09-01 00:00:00,US,20.0,0,enterprise,2024-03-01,False
6,A-1006,Foxtrot,2023-05-11 00:00:00,us,3.0,270,PRO,2024-03-02,False
13,A-1012,Lima,2023-11-11 00:00:00,US,1000000.0,90,pro,2024-03-02,False
5,A-1005,Echo,2023-04-01 00:00:00,US,8.0,-99,pro,2024-03-03,False
15,A-1014,NaN,2024-01-15 00:00:00,US,9.0,810,pro,2024-03-03,False
8,A-1008,Helio,2023-07-22 00:00:00,US,NaN,540,pro,2024-03-04,False


Missing business information:
I need a reliable record/version timestamp or another business rule that identifies which record is the authoritative latest state for each account before removing any repeated account records.

Question 7: One column, several disease

In [ ]:
# calculating mrr per seat using:
df['mrr_per_seat'] = df['mrr'] / df['seats']
df[['account_id', 'mrr_per_seat']]

,account_id,mrr_per_seat
0,A-1001,100.00000
1,A-1002,90.00000
2,A-1001,100.00000
3,A-1003,NaN
4,A-1004,99.00000
5,A-1005,-12.37500
6,A-1006,90.00000
7,A-1007,120.00000
8,A-1008,NaN
9,A-1009,90.00000


| Rows | Seats | mrr | mrr per seats | Possible cause | Actions |
| ---- | ----- | ------- | --------- | -------------- | --------- |
| A-1005 | 8 | -99 | -12.375 | The negative MRR is suspicious because -99 may be a special missing-value or error code rather than an actual negative price. | Investigate the meaning of -99 and convert it to NaN only after confirming the coding convention. |
| A-1008 | 540 | 900 | 1.67 | The number of seats is very high relative to the MRR, producing an unusually low MRR per seat. This could be an incorrect seat value or an unusual pricing arrangement. | Investigate the source of the seat count and the customer's billing arrangement Before changing the value. |
|A-1010 | 20 | 0 | 0 | An account with 20 seats and zero MRR is unusual. It could be a legitimate free/promotional account, or the MRR could be missing or incorrectly recorded as zero. | Investigate the customer's billing/subscription record before deciding whether the zero is valid. |
| A-1012 | 1,000,000 | 90 | 0.00009 | One million seats is extremely unusual compared with the other records and produces an implausibly small MRR per seat. This could be a data-entry or unit error. | Verify the original seat value and its unit before changing it. |

Two competing explanations

A-1010 has two explanations that the data alone cannot distinguish:

1. The account could legitimately have 20 seats but zero MRR, for example because it is free or promotional.
2. The MRR could be missing or incorrectly recorded as zero.

I would need the account's billing/subscription information to determine which explanation is correct. Therefore, I would not change the value based only on the MRR-per-seat calculation.

Important observation

A value being unusual does not automatically mean it is incorrect. For example, A-1012's one-million-seat value is highly suspicious, but I would verify the source before changing it. Similarly, A-1010's zero MRR may be legitimate. My approach is therefore to investigate the business meaning of suspicious values before cleaning them, rather than automatically removing statistical outliers.

Question 8: Parsing is not validating

In [ ]:
# The following was used to parse the columns successfully; this does not mean that all the dates are logically correct.
# I investigated the relationship between the two dates and found the following problems.
df['signup_ts'] = pd.to_datetime(df['signup_ts'], errors='coerce')
df['last_active'] = pd.to_datetime(df['last_active'], errors='coerce')

| Account | Problem | Explanation | Proposed fix |
| ------- | ------ | ----------- | ------------ |
|A-1007 | last_active occurs before signup_ts |	The account has a signup date of 2023-06-30, but its last activity is recorded as 2023-06-15, which is 15 days before signup. This is logically inconsistent if last_active represents activity after the account was created. |	Verify the original dates in the source system and correct the erroneous date. If it cannot be verified, replace the invalid value with NaT rather than inventing a date.
| A-1013 |	Implausible future last_active date |	The last_active value is 2099-01-01. Although pandas can parse it as a valid datetime, it is implausible compared with the 2023–2024 dates in the dataset. |	Verify the source value. If the correct date cannot be recovered, mark it as NaT or otherwise flag it as invalid.
| Timezone information |	Timestamps are timezone-naive |	The timestamps contain date and time information but no timezone/offset. Therefore, the data does not establish whether the times represent UTC, local time, or another timezone. |	I would not invent a timezone. I would confirm the source system's timezone. If the source is UTC, I would convert the timestamps to timezone-aware UTC values and use that convention consistently.

In [ ]:
# This code returned A-1007, confirming that its last_active date occurs before its signup date
df[df['last_active'] < df['signup_ts']]

,account_id,company,signup_ts,country,seats,mrr,plan,last_active,churned,mrr_per_seat
7,A-1007,Gamma,2023-06-30,DE,15.0,1800,enterprise,2023-06-15,True,120.0


In [ ]:
# This code which returned A-1013, confirming the suspicious 2099-01-01 value.
df[df['last_active'].dt.year > 2025]

,account_id,company,signup_ts,country,seats,mrr,plan,last_active,churned,mrr_per_seat
14,A-1013,Mike,2023-12-25,US,4.0,360,pro,2099-01-01,False,90.0


## Section D: Judgement

Question 9: The same null, two fates

In [ ]:
# Missing values in the company column
df[df['company'].isna()]

,account_id,company,signup_ts,country,seats,mrr,plan,last_active,churned
15,A-1014,NaN,2024-01-15 00:00:00,US,9.0,810,pro,2024-03-03,False


In [ ]:
# Missing values in the seats column
df[df['seats'].isna()]

,account_id,company,signup_ts,country,seats,mrr,plan,last_active,churned
8,A-1008,Helio,2023-07-22 00:00:00,US,NaN,540,pro,2024-03-04,False


The row for A-1014 has a missing value in the label column company, while A-1008 has a missing value in the measure column seats. I would not automatically drop either row because the appropriate treatment depends on the downstream use. A-1014 can still be useful for numerical analyses such as MRR or churn analysis even without a company name, whereas A-1008 cannot be included in a revenue-per-seat calculation because seats is required for the calculation. Therefore, dropping the A-1008 row could be appropriate for a revenue-per-seat analysis, while dropping A-1014 would unnecessarily remove otherwise usable data.

Question 10: Cleaning is a function of the question

For the first cleaning decision, I previously treated the two A-1002 records as requiring further investigation rather than simply removing one with drop_duplicates(), because they contain different values and may represent a legitimate account update. However, if the goal were to create a current-state customer dataset, I would reverse that decision and keep only the most recent A-1002 record, because retaining both would cause the same account to be represented twice.

For the second decision, I treated A-1005's -99 MRR as suspicious and something that should not automatically be accepted as a normal revenue value. However, if the goal were to analyze billing adjustments or financial corrections, I would reverse that decision and retain the value until its business meaning was confirmed, because negative amounts could potentially represent legitimate adjustments.

Conclusion: A dataset is not inherently "clean" or "dirty"; whether a value needs cleaning depends on the question the data is being used to answer. Cleaning is therefore a context-dependent judgement about whether the data is fit for a particular purpose, rather than a permanent property of the dataset.